# Agentic MRO Optimization - Complete Pipeline

This notebook demonstrates the integration of agentic mobility generation with MRO (Mobility Robustness Optimization).

**Pipeline:**
1. **Natural Language → Mobility Simulation**: Generate UE mobility data from text queries
2. **Topology Generation**: LLM-generated cell tower placement
3. **Agentic MRO Optimization**: Intelligent parameter optimization using multi-agent LLM workflow

---

## Part 0: Environment Setup Instructions

### Step 1: Create Virtual Environment

```bash
# Navigate to project root
cd /path/to/AgenticMaveric

# Create virtual environment
python3 -m venv .venv # python v3.10.16 recommened

# Activate (macOS/Linux)
source .venv/bin/activate

# Activate (Windows)
.venv\Scripts\activate
```

### Step 2: Install Requirements

```bash
pip install -r radp/digital_twin/requirements.txt

pip install -r notebooks/requirements.txt

pip install -r requirements-dev.txt

pip install -r apps/mobility_robustness_optimization/agentic_mro/requirements.txt
```

### Step 3: Configure API Keys

**For Agentic Mobility:**
```bash
# Copy example environment file
cp radp/digital_twin/agentic_mobility/.env.example .env

# Edit the .env file and add your API key
# AWS_ACCESS_KEY_ID
# AWS_SECRET_ACCESS_KEY
```

**For Agentic MRO:**
```bash

# Edit the .env file and add your API key in the apps/mobility_robustness_optimization/agentic_mro/config.yaml file
# in the following:
api_key: "YOUR_KEY_THERE"

# [RECOMMENDED] Edit the .env file and add your API key
# AWS_ACCESS_KEY_ID
# AWS_SECRET_ACCESS_KEY
```

### Step 4: Launch Jupyter

```bash
# From project root
jupyter notebook
```

Now you're ready to run this notebook!

---

## Part 1: Introduction & Setup

### What is Agentic MRO?

Agentic MRO combines two powerful AI-driven features:

**1. Agentic Mobility Generation**
- Transform natural language → mobility simulation
- LLM-powered parameter inference
- Automatic geocoding and validation

**2. Agentic MRO Optimization (Coming Soon)**
- Multi-agent LLM workflow for parameter optimization
- Intelligent search instead of brute-force
- Minimizes Radio Link Failures (RLFs) and handover interruptions
- Optimizes Hysteresis and Time-to-Trigger parameters

### What This Notebook Demonstrates:

**Current Implementation:**
1. Natural language → mobility simulation
2. Cell tower topology generation
3. Data preparation for MRO optimization

**Agentic MRO:**
4. Agentic MRO multi-agent optimization
5. Before/after parameter comparison
6. Performance metrics visualization

---

In [ ]:
# Setup Python path
import sys
from pathlib import Path

# Add maveric root to path
notebook_dir = Path.cwd()
maveric_root = notebook_dir.parent
if str(maveric_root) not in sys.path:
    sys.path.insert(0, str(maveric_root))

print(f"Notebook directory: {notebook_dir}")
print(f"Maveric root: {maveric_root}")

In [ ]:
# Create output directory
output_dir = Path("data/agentic_data/mro")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directory created: {output_dir}")
print(f"  Absolute path: {output_dir.resolve()}")
print(f"  Directory exists: {output_dir.exists()}")

In [ ]:
# Import required modules
import json
import pandas as pd

from radp.digital_twin.agentic_mobility.integration import AgenticMobilityIntegration
from radp.digital_twin.agentic_mobility.topology_generator import TopologyGenerator

print("All imports successful!")

---
## Part 2: Natural Language → Mobility Generation

Generate realistic UE mobility data that will be used for MRO optimization.

The system will:
1. Parse the natural language query
2. Resolve the location to lat/lon bounds
3. Generate RADP-compatible parameters
4. Validate and self-correct if needed
5. Generate mobility simulation data

**Note**: This may take ~5-10 seconds due to LLM API calls and geocoding.

---

In [ ]:
# Define natural language query
# For MRO optimization, we want realistic mobility patterns
query = "Give me UE dataset for Toshima, Tokyo, Japan. consider it as a urban area with lots of vehicles such as cars and motorcycles which are moving extreamly fast, almost as if circling around the whole map in near infinite speeds. Have 30 total devices, esnure tick of 500. Deploy an optimal amount of cell towers with a range of 2 to 4 towers. Ensure optimial freqeuncy among all cell towers, where each tower MUST experience different fequencies. Try to generate data that forces handover"

print(f"Query: '{query}'")
print("\nProcessing...")
print("This may take 5-10 seconds (LLM + geocoding API calls)")

In [ ]:
# Generate mobility data from natural language
df, metadata = AgenticMobilityIntegration.generate_from_natural_language(query)

print(f"\nGenerated {len(df)} mobility points for {metadata['query_intent']['num_ues']} UEs")

In [ ]:
# Display key metadata
query_intent = metadata['query_intent']

print("="*70)
print("GENERATION METADATA")
print("="*70)
print(f"Location: {query_intent['location']}")
print(f"Scenario Type: {query_intent['scenario_type']}")
print(f"Number of UEs: {query_intent['num_ues']}")
print(f"Number of Ticks: {query_intent['num_ticks']}")
print(f"Retry Count: {metadata['retry_count']}")

# Handle ue_distribution structure
ue_dist = query_intent['ue_distribution']
source = ue_dist.get('source', 'unknown')
print(f"\nUE Distribution (Source: {source}):")

# Get distribution values (exclude 'source' key)
for ue_type, percentage in ue_dist.items():
    if ue_type != 'source':
        print(f"  - {ue_type}: {percentage:.1%}")

print("="*70)

In [ ]:
# Display DataFrame preview
print("\nDataFrame Preview:")
display(df.head(10))

print(f"\nDataFrame Info:")
print(f"  - Total rows: {len(df)}")
print(f"  - Columns: {list(df.columns)}")
print(f"  - Unique UEs: {df['mock_ue_id'].nunique()}")
print(f"  - Ticks: {df['tick'].min()} to {df['tick'].max()}")
print(f"  - Lat range: [{df['lat'].min():.6f}, {df['lat'].max():.6f}]")
print(f"  - Lon range: [{df['lon'].min():.6f}, {df['lon'].max():.6f}]")

In [ ]:
# Save mobility data
csv_path = output_dir / "ue_mobility_data.csv"
metadata_path = output_dir / "ue_mobility_metadata.json"

df.to_csv(csv_path, index=False)
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved mobility data to: {csv_path}")
print(f"Saved metadata to: {metadata_path}")

---
## Part 3: Topology Generation

Generate cell tower topology using LLM-based intelligent placement.

The `TopologyGenerator` uses an LLM to:
- Understand the area type (urban/suburban/rural)
- Calculate optimal number of cell sites
- Place towers intelligently within spatial bounds
- Generate realistic cell configurations

This topology will be used by the MRO optimizer to calculate handover parameters.

---

In [ ]:
# Extract location data
location_data = metadata.get('location_data', {})
query_intent = metadata['query_intent']

# Generate topology using LLM
print("Generating cell tower topology using LLM...")
print("This may take a few seconds...\n")

topology_df = TopologyGenerator.generate_from_llm(
    raw_query=query_intent.get('raw_query', query),
    area_type=location_data.get("area_type", "suburban"),
    num_ues=query_intent.get("num_ues", 50),
    min_lat=location_data.get("min_lat", df["lat"].min()),
    max_lat=location_data.get("max_lat", df["lat"].max()),
    min_lon=location_data.get("min_lon", df["lon"].min()),
    max_lon=location_data.get("max_lon", df["lon"].max()),
    mobility_df=df,
    use_genetic_algorithm=True
)

print(f"Generated {len(topology_df)} cell sectors")

In [ ]:
# Display topology preview
print("\nTopology Preview:")
display(topology_df.head(10))

# Count unique cell sites
unique_locations = topology_df.groupby(["cell_lat", "cell_lon"]).size()
print(f"\nGenerated {len(unique_locations)} unique cell sites")

In [ ]:
# Save topology
topology_path = output_dir / "cell_topology.csv"
topology_df.to_csv(topology_path, index=False)

# Save generation parameters
params_info = {
    "query": query,
    "location_data": location_data,
    "query_intent": query_intent,
    "metadata": metadata,
}
params_path = output_dir / "generation_params.json"
with open(params_path, 'w') as f:
    json.dump(params_info, f, indent=2)

print(f"Saved topology to: {topology_path}")
print(f"Saved generation parameters to: {params_path}")

In [ ]:
# Validate spatial boundaries
print("\nBoundary Validation:")
print("="*70)
print(f"  UE Lat range: [{df['lat'].min():.6f}, {df['lat'].max():.6f}]")
print(f"  UE Lon range: [{df['lon'].min():.6f}, {df['lon'].max():.6f}]")
print(f"  Cell Lat range: [{topology_df['cell_lat'].min():.6f}, {topology_df['cell_lat'].max():.6f}]")
print(f"  Cell Lon range: [{topology_df['cell_lon'].min():.6f}, {topology_df['cell_lon'].max():.6f}]")

# Check towers within UE bounds
towers_in_bounds = topology_df[
    (topology_df['cell_lat'] >= df['lat'].min()) & 
    (topology_df['cell_lat'] <= df['lat'].max()) &
    (topology_df['cell_lon'] >= df['lon'].min()) & 
    (topology_df['cell_lon'] <= df['lon'].max())
]
print(f"\n  Cell towers within UE bounds: {len(towers_in_bounds)}/{len(topology_df)}")
print("="*70)

---
## Data Summary for MRO Optimization

The generated mobility and topology data is now ready for MRO optimization.

> for detailed visualization and more, please checkout `notebooks/agentic_mobility_model.ipynb`

Below is a summary of the data that will be used:

---

In [ ]:
# Print UE DataFrame summary
print("="*70)
print("UE MOBILITY DATA SUMMARY")
print("="*70)
print(f"Total UEs: {df['mock_ue_id'].nunique()}")
print(f"Total Ticks: {df['tick'].max() + 1}")
print(f"Total Mobility Points: {len(df)}")
print(f"\nSpatial Coverage:")
print(f"  Latitude: {df['lat'].min():.6f} to {df['lat'].max():.6f}")
print(f"  Longitude: {df['lon'].min():.6f} to {df['lon'].max():.6f}")
print(f"\nColumns: {list(df.columns)}")
print("\nSample data:")
display(df.head())

print("\n" + "="*70)
print("CELL TOPOLOGY SUMMARY")
print("="*70)
print(f"Total Cell Sectors: {len(topology_df)}")
print(f"Unique Cell Sites: {len(unique_locations)}")
print(f"\nSpatial Coverage:")
print(f"  Latitude: {topology_df['cell_lat'].min():.6f} to {topology_df['cell_lat'].max():.6f}")
print(f"  Longitude: {topology_df['cell_lon'].min():.6f} to {topology_df['cell_lon'].max():.6f}")
print(f"\nColumns: {list(topology_df.columns)}")
print("\nSample data:")
display(topology_df.head())

print("\n" + "="*70)
print("DATA READY FOR MRO OPTIMIZATION")
print("="*70)
print(f"Mobility CSV: {csv_path}")
print(f"Topology CSV: {topology_path}")
print(f"Metadata JSON: {metadata_path}")
print("="*70)

---
## Agentic MRO Optimization

**Multi-Agent LLM Workflow for Intelligent Parameter Optimization**

The agentic MRO feature uses a multi-agent LLM system to optimize MRO parameters:

1. **Analyzer Agent**: Analyzes network simulation data (signal quality, mobility patterns, handover risks)
2. **Strategy Agent**: Recommends optimal parameter ranges (Hysteresis, Time-to-Trigger)
3. **Coordinator Agent**: Iteratively suggests parameters, learns from history
4. **Finalize Agent**: Compiles results and outputs best parameters

**Implementation Status:**
- Core agentic MRO code is available at: `apps/mobility_robustness_optimization/agentic_mro/`

---

In [ ]:
import os

import pandas as pd
import yaml
from apps.mobility_robustness_optimization.agentic_mro.main import run_agentic_mro

from apps.mobility_robustness_optimization.agentic_mro.utils.visuals import (
    add_sinr_column,
    mro_plot_scatter,
    plot_sinr_db_by_ue,
)
from notebooks.radp_library import preprocess_ue_data
from radp.digital_twin.utils.cell_selection import perform_attachment_hyst_ttt
from radp.digital_twin.utils.constants import RLF_THRESHOLD

In [ ]:
def load_config(config_path="config.yaml"):
    """Load configuration from YAML file."""
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    return config

### Run Agentic MRO using config.yaml settings.
Load configuration from the Agentic MRO 


In [ ]:
config_path = maveric_root / "apps/mobility_robustness_optimization/agentic_mro/config.yaml"
config = load_config(config_path)

Get Default Provider Settings

In [ ]:
default_provider = config.get("default_provider", "groq")
provider_config = config["providers"][default_provider]

opt_config = config.get("optimization", {})

Get data paths


In [ ]:
data_config = config.get("data", {})
csv_path_from_config = data_config.get("input_csv")
topology_csv_from_config = data_config.get("topology_csv")

### Running Agentic MRO with config.yaml settings

In [ ]:
print("=" * 70)
print("Running Agentic MRO with config.yaml settings")
print("=" * 70)
print(f"Provider: {default_provider}")
print(f"Model: {provider_config.get('model')}")
print(f"Mobility Data: {csv_path_from_config}")
print(f"Topology Data: {topology_csv_from_config}")
print(f"Max Iterations: {opt_config.get('max_iterations', 3)}")
print("=" * 70 + "\n")

Check if API key is set


In [ ]:
if provider_config.get("AWS_ACCESS_KEY_ID") == "PUT_YOUR_BEDROCK_AWS_ACCESS_KEY_ID_HERE":
    print("⚠️  WARNING: API key not set in config.yaml!")
    print("Please edit config.yaml and replace 'UT_YOUR_BEDROCK_AWS_ACCESS_KEY_ID_HERE' with your actual Groq API key.")

## Preprocessing Data

In [ ]:
# Injecting constant RLF Threshold
opt_config["rlf_threshold"] = RLF_THRESHOLD

# Reading Data
ue_data = pd.read_csv(csv_path_from_config)
topology = pd.read_csv(topology_csv_from_config)

if ue_data.empty or topology.empty:
    raise ValueError("Input data is empty. Please check the CSV files.")

ue_data = ue_data.rename(columns={"lat": "latitude", "lon": "longitude"})

print("Successfully loaded UE mobility and topology data.")

sim_data = preprocess_ue_data(ue_data, topology)
sim_data = sim_data.rename(
    columns={"longitude": "loc_x", "latitude": "loc_y", "mock_ue_id": "ue_id", "cell_rxpwr_dbm": "cell_rxpower_dbm"}
)

sim_data = add_sinr_column(sim_data)
sim_data.to_csv("data/agentic_data/mro/sim_data.csv", index=False)

ticks = len(sim_data["tick"].unique())
# Assuming each tick represents 50ms (this value may need to be adjusted based on actual data characteristics)
tick_duration_seconds = 1  # 1 second per tick
T = ticks * tick_duration_seconds

# Injecting target score based on total simulation time
opt_config["target_score"] = T

print("\n" + "=" * 70)
print("Data Preprocessing Successfully Completed.")
print("=" * 70)

## Run Optimization

In [ ]:
result = run_agentic_mro(
        csv_path="data/agentic_data/mro/sim_data.csv",
        llm_config=provider_config,
        target_score=opt_config.get("target_score", 0.80),
        max_iterations=opt_config.get("max_iterations", 3),
        rlf_threshold=opt_config.get("rlf_threshold", -4.0),
    )

In [ ]:
print("=" * 70)
print(f"Best Hysteresis: {result['best_hysteresis']:.4f} dB")
print(f"Best TTT: {result['best_ttt']} ticks")
print(f"Best Score: {result['best_score']:.4f}")
print(f"Total Iterations: {result['total_iterations']}")
print("=" * 70)

## Create Visuals

In [ ]:
print("\n" + "=" * 70)
print("Creating Visualizations")
print("=" * 70)

# Generate optimal data with best parameters
optimal_data = perform_attachment_hyst_ttt(
    sim_data, hyst=result["best_hysteresis"], ttt=result["best_ttt"], rlf_threshold=RLF_THRESHOLD
)

# Create scatter plot (saved to file)
mro_plot_scatter(
    optimal_data, topology, save_path="data/agentic_data/mro/plots/mro_plot_scatter.png"
)
print("✓ Scatter plot saved to: data/agentic_data/mro/plots/mro_plot_scatter.png")

# Create interactive SINR plot for all UEs (returns Plotly figure)
unique_ue_ids = optimal_data["ue_id"].unique().tolist()
sinr_fig = plot_sinr_db_by_ue(
    optimal_data,
    sim_data,
    ue_ids=unique_ue_ids,
    figsize=(1400, 800),
    rlf_threshold=RLF_THRESHOLD,
)
print(f"✓ Interactive SINR plot created for {len(unique_ue_ids)} UEs")

print("\n" + "=" * 70)
print("Visualization Creation Complete")
print("=" * 70)

### Scatter Plot: Cell Towers and UE Locations

This plot shows:
- **Triangle markers**: Cell tower locations (color-coded by cell ID)
- **Circular markers**: UE positions (color-coded by connected cell)
- **Grey markers**: UEs experiencing RLF (Radio Link Failure)

In [ ]:
from IPython.display import Image, display

# Display the saved scatter plot
display(Image("data/agentic_data/mro/plots/mro_plot_scatter.png"))

### Interactive SINR Plot: Per-UE Analysis

This interactive plot shows SINR (Signal-to-Interference-plus-Noise Ratio) over time for each UE:
- **Dropdown menu**: Select different UEs to view their SINR performance
- **Dotted lines**: All candidate cell SINR values (context)
- **Solid lines**: Connected cell SINR (color-coded by cell ID)
- **Black X markers**: RLF (Radio Link Failure) events
- **Dashed line**: RLF threshold

Use the dropdown at the top-left to switch between UEs and explore their handover patterns.

In [ ]:
# Display the interactive SINR plot
sinr_fig.show()